# Generation of Background Events

## Load Libraries

In [1]:
import pandas as pd
import os
import pythia8

## Function that Generates Events for fixed Energy 

In [2]:
def generate_events(energy, nevent=100000, level='hadron', target='W', process_type='CC_numu'):
        
    # Setup Pythia
    pythia = pythia8.Pythia()

    # Set random seeds
    pythia.readString("Random:setSeed=on")
    pythia.readString("Random:seed=0")

    # Define beams (frameType = 2 for fixed target collisions)
    pythia.readString("Beams:frameType = 2")
    #pythia.readString("Beams:idA = 14")
    pythia.readString("Beams:eA = "+str(energy))
    pythia.readString("Beams:eB = 0")

    # Use the nuclear PDF nCTEQ15 for Tungsten (A=184, Z=74)
    pdffile = "nCTEQ15FullNuc_56_26_0000.dat"
    pythia.readString("PDF:pSet=lhagrid1:"+pdffile)
    #pdfpath = "/Users/felixkling/work/LHAPDF-6.2.1/installed/share/LHAPDF/nCTEQ15FullNuc_184_74/"
    if target == 'W': pdffile = "nCTEQ15FullNuc_184_74_0000.dat"
    elif target == 'Fe': pdffile = "nCTEQ15FullNuc_56_26_0000.dat"
    #pythia.readString("PDF:pSet=LHAGrid1:"+pdfpath+pdffile);

    # Activate necessary options for Pythia
    pythia.readString("PDF:lepton = off")
    pythia.readString("TimeShower:QEDshowerByL = off")

    # Fix Renormalization/Factorization scale
    pythia.readString("SigmaProcess:factorScale2=6")
    pythia.readString("SigmaProcess:renormScale2=6")

    # Minimal phase space cuts
    pythia.readString("PhaseSpace:mHatMin=1")
    pythia.readString("PhaseSpace:pTHatMin=0")
    pythia.readString("PhaseSpace:pTHatMinDiverge=0")

    # Define process type and set the beam accordingly
    if process_type == "CC_numu":
        pythia.readString("WeakBosonExchange:ff2ff(t:W)=on")
        pythia.readString("Beams:idA = 14")
    elif process_type == "CC_nuebar":
        pythia.readString("WeakBosonExchange:ff2ff(t:W)=on")
        pythia.readString("Beams:idA = -12")  
    elif process_type == "NC_numu":
        pythia.readString("WeakBosonExchange:ff2ff(t:gmZ)=on")
        pythia.readString("Beams:idA = 14")
    elif process_type == "NC_nuebar":
        pythia.readString("WeakBosonExchange:ff2ff(t:gmZ)=on")
        pythia.readString("Beams:idA = -12")  
    else:
        print("Error: process_type = " + process_type + " is not valid.")
        return pd.DataFrame()
    
    # Decide if shower and hadronization should be included
    if level == "hard":
        pythia.readString("HadronLevel:all=off")
        pythia.readString("PartonLevel:all=off")
    elif level == "parton":
        pythia.readString("HadronLevel:all=off")
        pythia.readString("PartonLevel:all=on")
    elif level == "hadron":
        pythia.readString("HadronLevel:all=on")
        pythia.readString("PartonLevel:all=on")
    else:
        print("Error: level = "+level+" is not a valid option")
        return pd.DataFrame()

    # Initialize
    pythia.init()

    # List to store events data for DataFrame
    event_data = []

    # Loop over events
    for ievent in range(nevent):
        
        # Generate next event
        if not pythia.next(): continue
        particles = pythia.process if level == "hard" else pythia.event

        # Loop through particles in event
        iiparticle = 0
        for iparticle, particle in enumerate(particles):
            
            # Reject non-final state particles
            if particle.status() < 0: continue
            # Collect particle data
            pid = particle.id()
            px, py, pz, e = particle.px(), particle.py(), particle.pz(), particle.e()
            parent_pid = particle.mother1()  

            # Append particle data as a new row
            event_data.append([ievent, iiparticle, energy, pid, px, py, pz, e, parent_pid])
            iiparticle+=1

    # Create output directory corresponding to different process
    output_dir = f"output_events_{process_type}"
    if not os.path.isdir(output_dir): os.makedirs(output_dir)
    #output_dir = pathlib.Path("output_events")
    #output_dir.mkdir(exist_ok=True)
    
    # Convert collected event data to DataFrame
    columns = ["ievent", "iparticle", "truth_energy", "pid", "px", "py", "pz", "e", "parent_pid"]
    df = pd.DataFrame(event_data, columns=columns)
    
    # See if data frame already exist, if yes, merge. 
    csv_file = output_dir+"/events_"+str(energy)+".csv"
    if os.path.exists(csv_file): 
        df_old = pd.read_csv(csv_file)
        nevt_old = df_old['ievent'].max()+1
        df['ievent'] = df['ievent']+ nevt_old
        df = pd.concat([df_old, df]) 
        
    # Export
    df.to_csv(csv_file, index=False)
    print("Saved events for energy "+str(energy)+" to "+csv_file)
    

## Use Function to Generate 100 Events per Energy

In [3]:
# List of energies
energies = [
    14.219093, 17.900778, 22.535744, 28.370820, 35.716747, 44.964720, 56.607229,
    71.264279, 89.716412, 112.946271, 142.190930, 179.007775, 225.357437,
    283.708205, 357.167468, 449.647202, 566.072289, 712.642790, 897.164117,
    1129.462706, 1421.909302, 1790.077754, 2253.574373, 2837.082046, 3571.674683,
    4496.472021, 5660.722891, 7126.427896, 8971.641174
] 

# Loop through all energies
for energy in energies:
    generate_events(energy=energy, nevent=100, level='hadron', target='Fe', process_type='CC_numu')


 *------------------------------------------------------------------------------------* 
 |                                                                                    | 
 |  *------------------------------------------------------------------------------*  | 
 |  |                                                                              |  | 
 |  |                                                                              |  | 
 |  |   PPP   Y   Y  TTTTT  H   H  III    A      Welcome to the Lund Monte Carlo!  |  | 
 |  |   P  P   Y Y     T    H   H   I    A A     This is PYTHIA version 8.310      |  | 
 |  |   PPP     Y      T    HHHHH   I   AAAAA    Last date of change: 25 Jul 2023  |  | 
 |  |   P       Y      T    H   H   I   A   A                                      |  | 
 |  |   P       Y      T    H   H  III  A   A    Now is 05 Nov 2024 at 21:15:58    |  | 
 |  |                                                                              |  | 
 |  |   Program docu

## Generate 100 more Events for E=1421.909302

In [4]:
generate_events(energy=1421.909302, nevent=100, level='hadron', target='Fe', process_type='CC_numu')


 *------------------------------------------------------------------------------------* 
 |                                                                                    | 
 |  *------------------------------------------------------------------------------*  | 
 |  |                                                                              |  | 
 |  |                                                                              |  | 
 |  |   PPP   Y   Y  TTTTT  H   H  III    A      Welcome to the Lund Monte Carlo!  |  | 
 |  |   P  P   Y Y     T    H   H   I    A A     This is PYTHIA version 8.310      |  | 
 |  |   PPP     Y      T    HHHHH   I   AAAAA    Last date of change: 25 Jul 2023  |  | 
 |  |   P       Y      T    H   H   I   A   A                                      |  | 
 |  |   P       Y      T    H   H  III  A   A    Now is 05 Nov 2024 at 21:16:07    |  | 
 |  |                                                                              |  | 
 |  |   Program docu